# API Examples

This notebook contains 3 main examples to outline the use of the `actantial` package:
1. API
    1. Use API backend for open annotation
    2. Use API backend with predefined labelset
    3. Use API backend with additional variables
    4. Resume an API run
2. GPU 
    1. Standard GPU run
    2. Quantised GPU run
3. Analyse labels


In [7]:
import yaml
import pandas as pd

from actantial import OpenAIBackend, HuggingFaceBackend
from actantial import run_extract, load_annotations 
from actantial.config import ACTANTS


In [8]:
# load data, important: needs id and text columns
data = pd.read_csv("data/sample_news_articles.csv")
data.head(1)

,id,publish_date,media_name,url,retrieval_time,text,day
0,c1127d11dd9375ab3112d32853f374ae,2023-11-10,aljazeera.com,https://www.aljazeera.com/news/2023/11/10/hosp...,2024-05-02 21:37:55.238747,Israeli troops encircle four hospitals as Gaza...,10


## Example 1: API
Make sure you have a `.env` file with an OpenAI API key stored.

In [3]:
# initiate LLM backend
backend = OpenAIBackend(model_name="gpt-4o-mini")


### Example 1.1: API open

In [4]:
# define parameters and run extraction
run_extract(
    data=data,
    backend=backend,
    output_dir="output",
    template="base_prompt",
)

Timestamp: 	20260610_185457
Log: 		output/logs/gpt-4o-mini_base_prompt_20260610_185457.log
Files: 		output/actantial_models/gpt-4o-mini/base_prompt/20260610_185457


100%|██████████| 5/5 [00:12<00:00,  2.52s/it]


In [9]:
annotations = load_annotations(
    data=data,
    label_folder="./output/actantial_models/gpt-4o-mini/base_prompt/20260610_185457/",
)

annotations[['id', 'text'] + ACTANTS]


,id,text,Subject,Object,Sender,Receiver,Helper,Opponent
0,c1127d11dd9375ab3112d32853f374ae,Israeli troops encircle four hospitals as Gaza...,palestinian civilians,safety and medical care,israeli military,patients and displaced individuals seeking ref...,medical staff,israeli military
1,24b44f2cb5cae1182f5e0c7f6f5e5024,Al-Shifa Hospital director confirms deaths aft...,director mohammad abu salmiya,casualties (deaths) from al-buraq school,israeli army,al-shifa hospital,al-shifa hospital staff,israeli army
2,c9dc1e8f5c83c3926aa92b20e3ebc348,Israeli tanks surround four hospitals in Gaza ...,israel,gaza's hospitals,israel,palestinians seeking medical care,international humanitarian organizations,hamas
3,6d195f1a9e6d677be447d04c8e6139b5,"Israel says more than 100,000 Palestinians hav...",palestinians,safety and refuge in southern gaza,israeli forces,palestinians,un humanitarian office,israeli forces
4,1cc33c8f8870ea78e57c2145a42d055a,Israel’s military has claimed Hamas operates a...,israel's military,hamas's operational capabilities at hospitals,israel's military,palestinian civilians and the international co...,indonesian foreign ministry,hamas


### Example 1.2: API with labelset
To get consistent annotations, it is recommended to compile a set of labels that the model should apply to the data, instead of coming up with new labels for each datapoint. 

For our example case of the Israeli-Palestinian conflict in the news those labels include actors such as `Israel` and `Palestine`, and objects, such as `violence` or `peace`. These are saved in `.yaml` files. 

In [ ]:
with open("data/sample_actors.yaml") as f:
    actor_labels = yaml.safe_load(f)

with open("data/sample_objects.yaml") as f:
    object_labels = yaml.safe_load(f)

run_extract(
    data=data,
    backend=backend,
    output_dir="output",
    template="prompt_closed",
    actor_labels=actor_labels,
    object_labels=object_labels,
)

Timestamp: 	20260515_141014
Log: 		output/logs/gpt-4o-mini_prompt_closed_20260515_141014.log
Files: 		output/actantial_models/gpt-4o-mini/prompt_closed/20260515_141014


  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:15<00:00,  3.17s/it]


In [10]:
annotations = load_annotations(
    data=data,
    label_folder="./output/actantial_models/gpt-4o-mini/prompt_closed/20260515_141014/",
)

annotations[['id', 'text'] + ACTANTS]

,id,text,Subject,Object,Sender,Receiver,Helper,Opponent
0,c1127d11dd9375ab3112d32853f374ae,Israeli troops encircle four hospitals as Gaza...,palestine/palestinians,security/safety,israel/israelis,palestine/palestinians,None,idf
1,24b44f2cb5cae1182f5e0c7f6f5e5024,Al-Shifa Hospital director confirms deaths aft...,palestine/palestinians,violence,israel/israelis,palestine/palestinians,None,idf
2,c9dc1e8f5c83c3926aa92b20e3ebc348,Israeli tanks surround four hospitals in Gaza ...,palestine/palestinians,security/safety,israel/israelis,palestine/palestinians,None,idf
3,6d195f1a9e6d677be447d04c8e6139b5,"Israel says more than 100,000 Palestinians hav...",palestine/palestinians,security/safety,israel/israelis,palestine/palestinians,None,idf
4,1cc33c8f8870ea78e57c2145a42d055a,Israel’s military has claimed Hamas operates a...,israel/israelis,violence,idf,palestine/palestinians,None,hamas


### Example 1.3: API with additional variables

You can pass additional columns from your DataFrame as template variables using `template_columns`. Here we pass `media_name` (the outlet that published the article) and `publish_date` alongside the text. Both columns must be string dtype and must appear as `{{ media_name }}` and `{{ publish_date }}` in the template.


In [7]:
backend.show_template("prompt_open_variables")


Template 'prompt_open_variables.txt' for model 'gpt-4o-mini' (source: gpt-4o-mini):

According to the Actantial Model by Greimas with the actant label set ["Subject", "Object", "Sender", "Receiver", "Helper", "Opponent"], the actants are defined as follows:

* Subject: The character who carries out the action and desires the Object.
* Object: The character or thing that is desired and transfered.
* Sender: The character who initiates the action, controls the Object, and transfers it to the Receiver.
* Receiver: The character who receives the Object.
* Helper: The character who assists the Subject in achieving its goal.
* Opponent: The character who opposes the Subject in achieving its goal.

Based on the Actantial Model, please recognize the actants in the following text.

Published by: {{ media_name }}
Published on: {{ publish_date }}
Text: {{ text }}


Question: What are the main actants in the text? 
1. Identify the main Object in the text.
2. Identify the respective Subject.
2. Ide

In [6]:
run_extract(
    data=data.head(1),
    backend=backend,
    output_dir="output",
    template="prompt_open_variables",
    template_columns=["media_name", "publish_date"],
)

Timestamp: 	20260602_143753
Log: 		output/logs/gpt-4o-mini_prompt_open_variables_20260602_143753.log
Files: 		output/actantial_models/gpt-4o-mini/prompt_open_variables/20260602_143753


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:05<00:00,  5.71s/it]


### Example 1.4: Resume run

In case a run breaks, you can resume it at a later stage. For this, You need to match the original parameters (output_dir, model, template) and then pass the correct timestamp to `resume_timestamp`. This new run will scan the output directory and skip files that were already analysed. I continues writing to the original log file.

In [9]:
run_extract(
    data=data,
    backend=backend,
    output_dir="output",
    template="base_prompt",
    resume_timestamp="20260610_185457" 
)

Log: 		output/logs/gpt-4o-mini_base_prompt_20260610_185457.log
Files: 		output/actantial_models/gpt-4o-mini/base_prompt/20260610_185457


100%|██████████| 5/5 [00:00<00:00, 5895.84it/s]


## Example 2: GPU
If you do not want to depend on company APIs, you can use HuggingFace to run models locally on your own GPU.

### Example 2.1: GPU standard

In [10]:
# initiate LLM backend
backend = HuggingFaceBackend(repository="google", model_name="gemma-3-1b-it", quantisation=False)


Loading model google/gemma-3-1b-it...
Model loaded successfully


In [11]:
# define parameters and run extraction
run_extract(
    data=data.head(2),
    backend=backend,
    output_dir="output",
    template="base_prompt"   
)

Timestamp: 	20260610_185737
Log: 		output/logs/gemma-3-1b-it_base_prompt_20260610_185737.log
Files: 		output/actantial_models/gemma-3-1b-it/base_prompt/20260610_185737


100%|██████████| 2/2 [02:59<00:00, 89.94s/it] 


In [11]:
annotations = load_annotations(
    data=data,
    label_folder="./output/actantial_models/gemma-3-1b-it/base_prompt/20260610_185737/",
)

annotations[['id', 'text'] + ACTANTS]

/Users/jelfes/ucd/actantial/actantial/io.py:290: UserWarning: Warning: 3/5 annotation files are missing.
  warnings.warn(


,id,text,Subject,Object,Sender,Receiver,Helper,Opponent
0,c1127d11dd9375ab3112d32853f374ae,Israeli troops encircle four hospitals as Gaza...,israeli troops,al-shifa hospital,israeli troops,al-shifa hospital,none,hamas
1,24b44f2cb5cae1182f5e0c7f6f5e5024,Al-Shifa Hospital director confirms deaths aft...,al-shifa hospital,deaths,al-shifa hospital,people,al-shifa hospital,israeli army
2,c9dc1e8f5c83c3926aa92b20e3ebc348,Israeli tanks surround four hospitals in Gaza ...,None,None,None,None,None,None
3,6d195f1a9e6d677be447d04c8e6139b5,"Israel says more than 100,000 Palestinians hav...",None,None,None,None,None,None
4,1cc33c8f8870ea78e57c2145a42d055a,Israel’s military has claimed Hamas operates a...,None,None,None,None,None,None


### Example 2.1: GPU quantised

To save VRAM, you can run models in 4bit quantisation using `bitsandbytes`. However, this only works for CUDA GPUs (no MPS unfotunately). Simply pass `quantisation=True` when initiating the backend.

In [11]:
# initiate LLM backend
backend = HuggingFaceBackend(repository="google", model_name="gemma-3-1b-it", quantisation=True)


Loading model google/gemma-3-1b-it...
Model loaded successfully


In [ ]:
run_extract(
    data=data,
    backend=backend,
    output_dir="output",
    template="base_prompt"   
)


Timestamp: 	20260611_092413
Log: 		output/logs/gemma-3-1b-it_base_prompt_20260611_092413.log
Files: 		output/actantial_models/gemma-3-1b-it/base_prompt/20260611_092413


100%|██████████| 5/5 [01:02<00:00, 12.70s/it]


In [12]:
annotations = load_annotations(
    data=data,
    label_folder="./output/actantial_models/gemma-3-1b-it/base_prompt/20260611_092413/",
)

annotations[['id', 'text'] + ACTANTS]

,id,text,Subject,Object,Sender,Receiver,Helper,Opponent
0,c1127d11dd9375ab3112d32853f374ae,Israeli troops encircle four hospitals as Gaza...,israeli troops,israeli tanks,israeli military forces,gaza ministry of health,gaza ministry of health,israeli military forces
1,24b44f2cb5cae1182f5e0c7f6f5e5024,Al-Shifa Hospital director confirms deaths aft...,al-shifa hospital,al-shifa hospital,al-shifa hospital,gazian,al-shifa hospital,gazian
2,c9dc1e8f5c83c3926aa92b20e3ebc348,Israeli tanks surround four hospitals in Gaza ...,hospital,hospital,israel,palestinians,doctors,israel
3,6d195f1a9e6d677be447d04c8e6139b5,"Israel says more than 100,000 Palestinians hav...",israel,israel,israel,palestinians,israel,palestinians
4,1cc33c8f8870ea78e57c2145a42d055a,Israel’s military has claimed Hamas operates a...,israel,al-shifa hospital,israel,al-shifa hospital,israeli military,palestinians


## Example 3: Analyse labels
The function `compare_annotations` is explained in detail in the [case study](https://jelfes.github.io/actantial/examples/case_study/).